# train.curated.v002-6: 長文分割（split-long）を反映した学習データ作成

目的:
- `train.curated.v002-4-2.xlsx` から **指定した `oare_id` を除外**したうえで、長文分割結果 `v002-5-split-long.json`（confidence が high/medium のみ）を結合し、新しい学習データ（v002-6）を作る。

入力:
- `data/curated/deep-past-initiative-machine-translation/train/train.curated.v002-4-2.xlsx`
- `notebooks/EDA/PROCESS-DATA/v002-5-split-long.json`

出力（例）:
- `data/curated/deep-past-initiative-machine-translation/train/train.curated.v002-6.xlsx`
- `data/curated/deep-past-initiative-machine-translation/train/train.curated.v002-6.csv`

注意:
- `OARE_IDS_TO_REMOVE` は base（v002-4-2）側の除外にのみ使い、split-long 側の採用（confidence フィルタ）には影響しません。
- `v002-5-split-long.json` 側に元 `oare_id` が入っていない場合は、結合後の `oare_id` に `split_id` を採用します（`oare_id` の意味が「元レコード」ではなく「学習行ID」になります）。
- 元 `oare_id` へ紐付けたい場合は、このノートブックの `RECORD_ID_TO_OARE_ID` か `RECORD_INDEX_TO_OARE_ID` を埋めてください。


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

from IPython.display import display

import numpy as np
import pandas as pd


In [2]:
def find_repo_root(start: Path | None = None, *, max_up: int = 8) -> Path:
    """notebooks/ 配下から実行しても repo root を見つけるためのヘルパー。"""
    cur = (start or Path.cwd()).resolve()
    for _ in range(max_up + 1):
        if (cur / "data" / "curated").exists() and (cur / "notebooks").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("repo root が見つかりませんでした（data/curated と notebooks を探索）")


REPO_ROOT = find_repo_root()
REPO_ROOT


PosixPath('/Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge')

In [3]:
# ===== 設定（ここだけ埋めれば動く想定） =====

BASE_XLSX_PATH = (
    REPO_ROOT
    / "data/curated/deep-past-initiative-machine-translation/train/train.curated.v002-4-2.xlsx"
)
SPLIT_JSON_PATH = REPO_ROOT / "notebooks/EDA/PROCESS-DATA/v002-5-split-long.json"

# v002-4-2 側から除外したい oare_id を列挙（split-long 側の採用とは独立）
OARE_IDS_TO_REMOVE: list[str] = [
    # 以下はあまりに<gap>などが多く、学習に悪影響がありそうなため学習からは取り除くレコード
    "02351a98-b66c-42eb-90a7-11d834afe8e8",
    "0960dbf7-bab1-4613-bb18-f81927fa3313",
    "3a251e5a-5fb9-4bad-8eda-ecc7cd57fcee"
]

# split-long の confidence フィルタ（要件: high/medium のみ）
ALLOWED_CONFIDENCE = {"high", "medium"}

# (任意) split-long の record_id → 元 oare_id の辞書（紐付けできるなら推奨）
# 例: {"record_1": "<uuid>", "record_2": "<uuid>"}
RECORD_ID_TO_OARE_ID: dict[str, str] = {}

# (任意) record_1, record_2, ... を「順番」で紐付けたい場合のリスト
# record_1 は [0], record_2 は [1] ... を参照
RECORD_INDEX_TO_OARE_ID: list[str] = []

OUTPUT_STEM = "train.curated.v002-6"
OUTPUT_DIR = REPO_ROOT / "data/curated/deep-past-initiative-machine-translation/train"
OUTPUT_DIR


PosixPath('/Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/curated/deep-past-initiative-machine-translation/train')

In [4]:
assert BASE_XLSX_PATH.exists(), BASE_XLSX_PATH
assert SPLIT_JSON_PATH.exists(), SPLIT_JSON_PATH

print("BASE_XLSX_PATH:", BASE_XLSX_PATH)
print("SPLIT_JSON_PATH:", SPLIT_JSON_PATH)
print("remove oare_id:", len(OARE_IDS_TO_REMOVE))


BASE_XLSX_PATH: /Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/curated/deep-past-initiative-machine-translation/train/train.curated.v002-4-2.xlsx
SPLIT_JSON_PATH: /Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/notebooks/EDA/PROCESS-DATA/v002-5-split-long.json
remove oare_id: 3


In [5]:
# openpyxl が必要（Kaggle/Jupyter では通常利用可能な想定）
df_base = pd.read_excel(BASE_XLSX_PATH)

# 先頭に "Unnamed: 0" などが居る場合は落とす
unnamed_cols = [c for c in df_base.columns if str(c).startswith("Unnamed")]
if unnamed_cols:
    df_base = df_base.drop(columns=unnamed_cols)

print("base rows:", len(df_base))
print("base cols:", df_base.columns.tolist())
display(df_base.head(3))


base rows: 1588
base cols: ['oare_id', 'label', 'excavation_no', 'transliteration', 'pdf', 'translation']


,oare_id,label,excavation_no,transliteration,pdf,translation
0,01cc1406-fc42-4252-aeae-6b4538b069c5,Cuneiform Tablet Kt 92/k 239 (AKT 5 57),Kt 92/k 239,17 GÍN KÙ.BABBAR ší-im e-ma-ri-im 30 ma-na URU...,5,"17 shekels of silver, the price of a donkey, 3..."
1,017c5be4-f459-4ef5-a045-5933e79c01f9,Cuneiform Tablet Kt 92/k 245 (AKT 5 69),Kt 92/k 245,ṭup-pu-um ša 10 ma-na KÙ.BABBAR ša PÚZUR-a-šur...,5,A tablet about 10 minas of silver of Puzur-Ass...
2,f52c21e6-5eb4-4a73-a539-c2686cc935b8,Cuneiform Tablet Kt 89/k 289 (AKT 5 76),Kt 89/k 289,a-na ku-li-a qí-bi4-ma um-ma a-šùr-i-mì-tí-ma ...,5,"Say to Kuliya, thus Aššur-imittī: In Šalatuwar..."


In [6]:
required_cols = {"oare_id", "transliteration", "translation"}
missing = required_cols - set(df_base.columns)
if missing:
    raise ValueError(f"BASE_XLSX に必須列が見つかりません: {sorted(missing)}")

df_base_filtered = df_base.copy()
if OARE_IDS_TO_REMOVE:
    before = len(df_base_filtered)
    df_base_filtered = df_base_filtered[~df_base_filtered["oare_id"].isin(OARE_IDS_TO_REMOVE)].copy()
    print("dropped base rows:", before - len(df_base_filtered))

print("base_filtered rows:", len(df_base_filtered))
if OARE_IDS_TO_REMOVE:
    assert not df_base_filtered["oare_id"].isin(OARE_IDS_TO_REMOVE).any()


dropped base rows: 3
base_filtered rows: 1585


In [7]:
split_raw = json.loads(SPLIT_JSON_PATH.read_text(encoding="utf-8"))
df_split = pd.DataFrame(split_raw)

print("split rows (raw):", len(df_split))
print("split cols:", df_split.columns.tolist())
display(df_split.head(3))

if "confidence" not in df_split.columns:
    raise ValueError("split-long JSON に confidence 列がありません")

conf_counts = df_split["confidence"].value_counts(dropna=False)
display(conf_counts)


split rows (raw): 1033
split cols: ['record_id', 'split_id', 'status', 'transliteration_chunk', 'translation_chunk', 'boundary_reason', 'confidence']


,record_id,split_id,status,transliteration_chunk,translation_chunk,boundary_reason,confidence
0,record_1,record_1__01,ok,um-ma kà-ru-um kà-ni-iš-ma a-na ša-qí-il5 da-t...,"Thus Kaniš, say to the -payers, our messenger ...",導入句＋命令文（Aššurの短剣を出せ）までで意味的にまとまる,high
1,record_1,record_1__02,ok,ik-ri-ba-am šu-uk-na-ma ma-ma-an KÙ.AN a-na ší...,and impose an oath and whoever has sold meteor...,課税・徴収命令の一連（tithe＋税＋Kuliyaが持参）で完結,high
2,record_1,record_1__03,ok,šu-ma i-qá-bi4 um-ma šu-ut-ma KÙ.AN ša a-wi-li...,If the person in question protests and declare...,条件文（抗議時の処理）＋誓約＋警告までで一まとまり,medium


confidence
high      895
medium    129
low         9
Name: count, dtype: int64

In [8]:
df_split = df_split[df_split["confidence"].isin(sorted(ALLOWED_CONFIDENCE))].copy()
print("split rows (filtered):", len(df_split))
display(df_split["confidence"].value_counts())

# base の列名に揃える（chunk → transliteration/translation）
if "transliteration_chunk" not in df_split.columns or "translation_chunk" not in df_split.columns:
    raise ValueError("split-long JSON に transliteration_chunk / translation_chunk が見つかりません")

df_split["transliteration"] = df_split["transliteration_chunk"].astype(str)
df_split["translation"] = df_split["translation_chunk"].astype(str)

# record_id → oare_id の補助
record_pat = re.compile(r"^record_(\d+)$")

def map_record_id_to_parent_oare_id(record_id: str) -> str | None:
    if record_id in RECORD_ID_TO_OARE_ID:
        return RECORD_ID_TO_OARE_ID[record_id]
    m = record_pat.match(record_id)
    if m and RECORD_INDEX_TO_OARE_ID:
        idx = int(m.group(1)) - 1
        if 0 <= idx < len(RECORD_INDEX_TO_OARE_ID):
            return RECORD_INDEX_TO_OARE_ID[idx]
    return None


df_split["oare_id_parent"] = df_split["record_id"].astype(str).map(map_record_id_to_parent_oare_id)

# oare_id の付与
# - parent が取れる: parent + "__" + split_suffix
# - 取れない: split_id をそのまま学習行IDとして使う
def build_oare_id(row: pd.Series) -> str:
    parent = row.get("oare_id_parent")
    split_id = str(row.get("split_id"))
    if isinstance(parent, str) and parent:
        # split_id 末尾 "__01" を拾う（無ければ split_id 全体を使う）
        m = re.search(r"__(\d+)$", split_id)
        suffix = m.group(1) if m else split_id
        return f"{parent}__{suffix}"
    return split_id


df_split["oare_id"] = df_split.apply(build_oare_id, axis=1)
df_split["source"] = "split_long"

print("split unique oare_id:", df_split["oare_id"].nunique(), "/", len(df_split))
display(df_split[["record_id", "split_id", "oare_id_parent", "oare_id", "confidence"]].head(10))


split rows (filtered): 1024


confidence
high      895
medium    129
Name: count, dtype: int64

split unique oare_id: 464 / 1024


,record_id,split_id,oare_id_parent,oare_id,confidence
0,record_1,record_1__01,None,record_1__01,high
1,record_1,record_1__02,None,record_1__02,high
2,record_1,record_1__03,None,record_1__03,medium
3,record_1,record_1__04,None,record_1__04,high
4,record_2,record_2__01,None,record_2__01,high
5,record_2,record_2__02,None,record_2__02,high
6,record_2,record_2__03,None,record_2__03,high
7,record_2,record_2__04,None,record_2__04,high
8,record_2,record_2__05,None,record_2__05,high
9,record_2,record_2__06,None,record_2__06,medium


In [9]:
# base 側にもメタ列を追加して揃える
df_base_out = df_base_filtered.copy()
df_base_out["source"] = "base"
for col in ["record_id", "split_id", "confidence", "boundary_reason", "status", "oare_id_parent"]:
    if col not in df_base_out.columns:
        df_base_out[col] = np.nan

# split 側を base の列順に寄せつつ、存在しない列は NaN で作る
df_split_out = df_split.copy()
for col in df_base_out.columns:
    if col not in df_split_out.columns:
        df_split_out[col] = np.nan

# base に無い split 由来列も残す（比較/追跡のため）
extra_split_cols = [c for c in df_split_out.columns if c not in df_base_out.columns]

# 連結
df_out = pd.concat(
    [
        df_base_out,
        df_split_out[df_base_out.columns.tolist() + extra_split_cols],
    ],
    ignore_index=True,
)

print("out rows:", len(df_out))
display(df_out.head(3))


out rows: 2609


,oare_id,label,excavation_no,transliteration,pdf,translation,source,record_id,split_id,confidence,boundary_reason,status,oare_id_parent,transliteration_chunk,translation_chunk
0,01cc1406-fc42-4252-aeae-6b4538b069c5,Cuneiform Tablet Kt 92/k 239 (AKT 5 57),Kt 92/k 239,17 GÍN KÙ.BABBAR ší-im e-ma-ri-im 30 ma-na URU...,5,"17 shekels of silver, the price of a donkey, 3...",base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,017c5be4-f459-4ef5-a045-5933e79c01f9,Cuneiform Tablet Kt 92/k 245 (AKT 5 69),Kt 92/k 245,ṭup-pu-um ša 10 ma-na KÙ.BABBAR ša PÚZUR-a-šur...,5,A tablet about 10 minas of silver of Puzur-Ass...,base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,f52c21e6-5eb4-4a73-a539-c2686cc935b8,Cuneiform Tablet Kt 89/k 289 (AKT 5 76),Kt 89/k 289,a-na ku-li-a qí-bi4-ma um-ma a-šùr-i-mì-tí-ma ...,5,"Say to Kuliya, thus Aššur-imittī: In Šalatuwar...",base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# ===== 簡易検証 =====

assert df_out["transliteration"].notna().all(), "transliteration に NaN が含まれます"
assert df_out["translation"].notna().all(), "translation に NaN が含まれます"

dup = df_out["oare_id"].astype(str).duplicated(keep=False)
if dup.any():
    print("WARNING: oare_id が重複しています（先頭のみ表示）")
    display(df_out.loc[dup, ["oare_id", "source", "record_id", "split_id"]].head(20))

if OARE_IDS_TO_REMOVE:
    assert not df_out["oare_id"].isin(OARE_IDS_TO_REMOVE).any(), "除外対象 oare_id が残っています"

print("source counts:")
display(df_out["source"].value_counts())


,oare_id,source,record_id,split_id
1558,NaN,base,NaN,NaN
1559,NaN,base,NaN,NaN
1560,NaN,base,NaN,NaN
1561,NaN,base,NaN,NaN
1562,NaN,base,NaN,NaN
1563,NaN,base,NaN,NaN
1564,NaN,base,NaN,NaN
1565,NaN,base,NaN,NaN
1566,NaN,base,NaN,NaN
1567,NaN,base,NaN,NaN


source counts:


source
base          1585
split_long    1024
Name: count, dtype: int64

In [11]:
# ===== 保存 =====

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_xlsx = OUTPUT_DIR / f"{OUTPUT_STEM}.xlsx"
out_csv = OUTPUT_DIR / f"{OUTPUT_STEM}.csv"

# Excel は openpyxl が必要（read_excel と同様）
df_out.to_excel(out_xlsx, index=False)
df_out.to_csv(out_csv, index=False)

print("wrote:", out_xlsx)
print("wrote:", out_csv)


wrote: /Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/curated/deep-past-initiative-machine-translation/train/train.curated.v002-6.xlsx
wrote: /Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/curated/deep-past-initiative-machine-translation/train/train.curated.v002-6.csv


In [12]:
df_out

,oare_id,label,excavation_no,transliteration,pdf,translation,source,record_id,split_id,confidence,boundary_reason,status,oare_id_parent,transliteration_chunk,translation_chunk
0,01cc1406-fc42-4252-aeae-6b4538b069c5,Cuneiform Tablet Kt 92/k 239 (AKT 5 57),Kt 92/k 239,17 GÍN KÙ.BABBAR ší-im e-ma-ri-im 30 ma-na URU...,5,"17 shekels of silver, the price of a donkey, 3...",base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,017c5be4-f459-4ef5-a045-5933e79c01f9,Cuneiform Tablet Kt 92/k 245 (AKT 5 69),Kt 92/k 245,ṭup-pu-um ša 10 ma-na KÙ.BABBAR ša PÚZUR-a-šur...,5,A tablet about 10 minas of silver of Puzur-Ass...,base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,f52c21e6-5eb4-4a73-a539-c2686cc935b8,Cuneiform Tablet Kt 89/k 289 (AKT 5 76),Kt 89/k 289,a-na ku-li-a qí-bi4-ma um-ma a-šùr-i-mì-tí-ma ...,5,"Say to Kuliya, thus Aššur-imittī: In Šalatuwar...",base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,aefa478d-c47d-42d2-a3b5-3f150603ccee,Cuneiform Tablet Kt 92/k 203 (AKT 5 2),Kt 92/k 203,um-ma kà-ru-um kà-ni-iš-ma a-na ša-qí-il5 da-t...,5,"Thus Kaniš, say to the -payers, our messenger ...",base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,55b115ac-6a6d-48e6-86df-ff449b4b346b,Cuneiform Tablet Kt 92/k 207 (AKT 5 5),Kt 92/k 207,um-ma kà-ru-um kà-ni-iš-ma a-na ku-li-a ša-qí-...,5,"Thus karum Kanesh, speak to Kuliya, the datu-p...",base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2604,akk_record_02__04,NaN,NaN,14 1/2 ma-na KB ṣa-ru-pu-um KI Puzur₄-A-šur DU...,NaN,14 1/2 minas of refined silver is owed by Puzu...,split_long,akk_record_02,akk_record_02__04,high,次の債務者（Puzur-Aššur）に関する金額・期限・証人のまとまりで区切れるため,ok,None,14 1/2 ma-na KB ṣa-ru-pu-um KI Puzur₄-A-šur DU...,14 1/2 minas of refined silver is owed by Puzu...
2605,akk_record_02__05,NaN,NaN,<gap> ma-na KB ṣa-ru-pu-um KI A-šur-ma-lik DUM...,NaN,<gap> minas of refined silver is owed by Aššur...,split_long,akk_record_02,akk_record_02__05,high,最後の債務（Aššur-malik）と総括文（Kaneshでの記録）までで全体が締めくくられるため,ok,None,<gap> ma-na KB ṣa-ru-pu-um KI A-šur-ma-lik DUM...,<gap> minas of refined silver is owed by Aššur...
2606,akk_record_01__01,NaN,NaN,um-ma kà-ru-um Kà-ni-iš-ma a-na ša-qí-il₅ da-t...,NaN,"Thus kārum Kaneš, say to the dātu-payers, our ...",split_long,akk_record_01,akk_record_01__01,high,導入句と最初の命令・条件文のまとまり（誓約まで）で意味的に完結しているため,ok,None,um-ma kà-ru-um Kà-ni-iš-ma a-na ša-qí-il₅ da-t...,"Thus kārum Kaneš, say to the dātu-payers, our ..."
2607,akk_record_01__02,NaN,NaN,ni-ša-me-ma 30 ANŠE Tù-i-mu-um DUMU A-šur ha-r...,NaN,"We hear that an Assyrian, Tūʾimum, has brought...",split_long,akk_record_01,akk_record_01__02,high,Tūʾimumの事例提示と一般化された規則のまとまりで一段落しているため,ok,None,ni-ša-me-ma 30 ANŠE Tù-i-mu-um DUMU A-šur ha-r...,"We hear that an Assyrian, Tūʾimum, has brought..."
